In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [3]:
load_dotenv(override=True)
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "qwen2.5-coder:3b"

In [4]:
openai = OpenAI()

In [5]:
# import urllib.request

# def get_book_from_gutenberg(url, output_file):
#     print("Downloading book...")
#     urllib.request.urlretrieve(url, output_file)
#     print(f"Saved successfully as {output_file}!")

In [6]:
# download the book from gutenberg

# get_book_from_gutenberg("https://www.gutenberg.org/cache/epub/75570/pg75570.txt", "psychology_of_stock_market.txt")

In [7]:
# # Get the text from the pdf file- use it as a context for the agent
# reader = PdfReader("/Users/macminived/vpvprojects/agents/1_foundations/Dev_Handson/Psychology_of_Stock_Market.pdf")
# psychology_sm = ""
# for page in reader.pages:
#     if page.page_number >=4:
#         text = page.extract_text(extraction_mode="layout")
#         if text:
#             psychology_sm += text

In [8]:
with open(f"/Users/macminived/vpvprojects/agents/1_foundations/Dev_Handson/psychology_of_stock_market.txt","r",encoding="utf-8") as f:
    psychology_sm = f.read()

In [9]:
with open(f"/Users/macminived/vpvprojects/agents/1_foundations/Dev_Handson/Market_Sentiment_Expert_Summary.txt","r",encoding="utf-8") as f:
    summary = f.read()

In [10]:
# print(psychology_sm[:500])

In [11]:
agent_name = "RJ"

In [13]:
# define the system prompt
system_prompt = f"""
# Your role

You are a helpful assistant named {agent_name} that can answer questions about the psychology of stock market on your website. You are given a summary of your role and background as well as \
a context of the psychology of stock market based on which you can answer questions from the user. 

Here are the details of your role and background:

Your job is to stay true to the context and answer as accurately as possible. Do NOT assume anything.\
TOOL USE INSTRUCTIONS:
1. ALWAYS refer to the Context you are provided with and the conversation history FIRST 
2. If the user's question can be answered using the provided context DO NOT CALL any tools
3. Use your tools to do a web search on live internet ONLY IF the answer is not available in the context.
"""
system_prompt+=f"\n\n## Summary:\n{summary}\n\n## Context:\n{psychology_sm}\n\n"
system_prompt+= f"With this information, answer as if the user is a potential client or future employer who came across the website."

In [30]:
print(system_prompt[:2000])


# Your role

You are a helpful assistant named RJ that can answer questions about the psychology of stock market on your website. You are given a summary of your role and background as well as a context of the psychology of stock market based on which you can answer questions from the user. 

Here are the details of your role and background:

Your job is to stay true to the context and answer as accurately as possible. Do NOT assume anything.TOOL USE INSTRUCTIONS:
1. ALWAYS refer to the Context you are provided with and the conversation history FIRST 
2. If the user's question can be answered using the provided context DO NOT CALL any tools
3. Use your tools to do a web search on live internet ONLY IF the answer is not available in the context.


## Summary:
I am RJ. I am a stock market sentiment expert who has been studying the market for over 10 years as a mathematician & statistician. 
I believe that the movements of prices on the exchanges are dependent to a very large degree on t

In [28]:
# define evaluator system prompt
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is of acceptable quality. \
The Agent is someone who can answer questions about the psychology of stock market on a website. \
The Agent has been provided with a role summary and context. Here's the information:"

evaluator_system_prompt+=f"\n\n## Summary:\n{summary}\n\n## Context:\n{psychology_sm}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [15]:
# Pydantic Model for evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [16]:
#Evaluate using other model- OpenAI

def evaluate(message,reply,history) -> Evaluation:
    messages = [{"role":"user","content":f"Message from user:{message}\n\n Reply from agent:{reply}\n\n Conversation history:{history}"}]
    response = openai.responses.parse(model="gpt-5-nano",instructions=evaluator_system_prompt,input=messages,text_format=Evaluation)
    evaluation_output = response.output_parsed
    
    return evaluation_output

In [17]:
def rerun(reply,message,history,feedback):
    messages = [{"role":"system","content":system_prompt+f"\n Your previous answer:{reply} got rejected.\n Feedback for the rejected answer:{feedback}"}]+history+[{"role":"user","content":message}]
    response = ollama.chat.completions.create(
        model=model_name,
        messages=messages,
        extra_body={
            "options": {
            "num_ctx": 8192  # Caps context window to 8k tokens (saves ~2-3 GB RAM)
            }
        }
    )
    reply = response.choices[0].message.content

    return reply

In [18]:
# A tool must be defined as the ollama uses local models; custom function tools supported     
tools = [{
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Use web search to check for answers online",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query term",
                }
            },
            "required": ["query"],
        },
    },
}]

In [35]:
# define chat function 
# using responses api
# using in built tools from ollama/openai. Imported web_search but passing the value using custom function
def chat(message,history):
    messages = [{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response = ollama.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=tools,
        extra_body={
            "options": {
            "num_ctx": 8192  # Caps context window to 8k tokens (saves ~2-3 GB RAM)
            }
        }
    )
    response_message = response.choices[0].message
    
    if response_message.tool_calls:
        messages.append(response_message)
        for item in response_message.tool_calls:
            if item.function.name == "web_search":
                args = json.loads(item.function.arguments)
                result = web_search(args.get("query",""))
                # Safely parse result: avoid sending empty dict/list string
                if not result:
                    content_result = (
                        "No web search results found. Proceed using your base"
                        " knowledge."
                    )
                elif isinstance(result, (dict, list)):
                    content_result = json.dumps(result)
                else:
                    content_result = str(result)
                print(f"Web search results:\n{content_result}")
                messages.append({
                    "role":"tool",
                    "content":content_result,
                    "tool_call_id":item.id
                })
        response = ollama.chat.completions.create(
            model=model_name,
            messages=messages,
            # tools = tools, # dont need a tool call here again
            extra_body={
                "options": {
                "num_ctx": 8192  # Caps context window to 8k tokens (saves ~2-3 GB RAM)
                }
            }
        ) 
        reply = response.choices[0].message.content
    else:
        reply = response.choices[0].message.content
        
    evaluation = evaluate(message,reply,history)

    if(evaluation.is_acceptable):
        print(f"Response from {agent_name} is acceptable")
        print(f"Feedback from evaluaor: {evaluation.feedback}")
    else:
        print(f"Response from {agent_name} is not acceptable")
        print(f"Feedback from evaluator: {evaluation.feedback}")
        print("Retrying...")
        reply = rerun(reply,message,history,evaluation.feedback)
    
    return reply


In [ ]:
# chat("Hi ! How does the market sentiment cause rise and fall in prices of a stock?",[])

In [ ]:
demo = gr.ChatInterface(chat, type="messages")

try:
  demo.launch()
except KeyboardInterrupt:
  demo.close()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Response from RJ is not acceptable
Feedback from evaluator: The agent did not answer the user's question. Instead, it returned a JSON object requesting a web search for 'stock market psychology' without providing any content. The user asked for factors affecting stock market psychology, so the response should have included a concise, structured list of such factors (e.g., macro news and policy signals, earnings and fundamentals, market microstructure and liquidity, behavioral biases like loss aversion and herd behavior, sentiment and media influence, discounting of future events, interest rates and liquidity conditions, global spillovers, and investor type differences). If a search is allowed, it should accompany a summarized synthesis rather than a bare search directive.
Retrying...
Response from RJ is not acceptable
Feedback from evaluator: The latest reply does not answer the user's question. It merely shows a web_search action payload and provides no content about factors affecting